# 14 · Beyond the Training Ground
**CAPSTONE** · ⏱ 15 min · 🧪 1 exercise

*Where to go next — real libraries, bigger problems, and the open frontiers of RL.*

> **First time?** Open **00 · START HERE** in the same folder. Then run the cell below to connect this notebook to your Drive folder — your progress and XP are saved there.

In [ ]:
#@title ▶ **Run me first** — connect this notebook to your Reward Signal folder
#@markdown Mounts Google Drive and finds the `RewardSignal` folder (progress, XP and your files live there).
#@markdown Leave `ROOT` empty unless you put the folder somewhere unusual, e.g. `/content/drive/MyDrive/courses/RewardSignal`.
ROOT = ""  #@param {type:"string"}
LESSON = "14-beyond"
import os, sys

def _rs_locate(root=""):
    tries = [root, os.environ.get("RS_ROOT", "")]
    try:
        from google.colab import drive  # only exists in Colab
        if not os.path.isdir("/content/drive/MyDrive"):
            drive.mount("/content/drive")
        base = "/content/drive/MyDrive"
        tries += [f"{base}/RewardSignal", f"{base}/Colab Notebooks/RewardSignal", f"{base}/RewardSignal/RewardSignal"]
        tries += [os.path.join(base, d, "RewardSignal") for d in sorted(os.listdir(base)) if not d.startswith(".")]
        tries += ["/content/RewardSignal"]
    except ImportError:  # local Jupyter: the folder this notebook sits in
        tries += [os.getcwd(), os.path.dirname(os.getcwd())]
    for t in tries:
        if t and os.path.isfile(os.path.join(t, "runtime", "rlcolab.py")):
            return os.path.abspath(t)
    raise FileNotFoundError("Couldn't find the RewardSignal folder. Upload it to My Drive "
                            "(see its README.md) or type its full path into ROOT, then run this cell again.")

RS_ROOT = _rs_locate(ROOT.strip())
sys.dont_write_bytecode = True  # no __pycache__ folders in your Drive
if os.path.join(RS_ROOT, "runtime") not in sys.path:
    sys.path.insert(0, os.path.join(RS_ROOT, "runtime"))
import rlcolab
rlcolab.setup(RS_ROOT, lesson=LESSON)

You have built every core idea of reinforcement learning by hand. That is the hard part.
Everything from here is scale, engineering, and a handful of new ideas layered on the same
foundations: **values, policies, bootstrapping, exploration**.

## 01 · The map you now hold

| Family | You built | Industrial-strength descendants |
|---|---|---|
| Bandits | ε-greedy, UCB, gradient bandit | Thompson sampling, contextual bandits (recommendation, ads, A/B testing) |
| Planning with a model | policy & value iteration, Dyna-Q | MCTS, AlphaZero, MuZero, model-based RL (Dreamer) |
| Model-free value-based | MC, SARSA, Q-learning, DQN | Rainbow, distributional RL (C51, QR-DQN), R2D2 |
| Policy gradient | REINFORCE, actor-critic, PPO | TRPO, IMPALA, RLHF (PPO / DPO for language models) |
| Continuous control | — | DDPG, TD3, **SAC** |

> **🔑 KEY IDEA** · **The one sentence to remember**
>
> Almost every RL algorithm is generalised policy iteration: evaluate the current behaviour
> a little, improve it a little, repeat — with different choices for *how* to estimate and *how*
> to improve.

## 02 · Use real libraries

Your own implementations are perfect for understanding. For real projects, reach for battle-tested
code. Install into the same Python that runs this app, then restart `server.py`:

```bash
pip install gymnasium torch stable-baselines3
```

**Gymnasium** is the standard environment API — the one you have been using all along:

```python
import gymnasium as gym
env = gym.make("CartPole-v1")
obs, info = env.reset(seed=0)
obs, reward, terminated, truncated, info = env.step(env.action_space.sample())
```

**Stable-Baselines3** gives you tuned PPO / DQN / SAC in three lines:

```python
from stable_baselines3 import PPO
model = PPO("MlpPolicy", "CartPole-v1", verbose=0).learn(total_timesteps=50_000)
```

> **💡 TIP** · **Wrap Core Runner**
>
> Core Runner already speaks the Gymnasium step API. Wrapping it as a real `gym.Env`
> (declare `observation_space` / `action_space`) lets you throw SB3's PPO at level 4 and compare
> it with your own DQN — a great weekend project. The sandbox below is a good place to start.

## 03 · Directions worth exploring

- **Continuous actions** — robotics-style control with SAC on `Pendulum-v1` or MuJoCo tasks.
- **Partial observability** — recurrent policies (LSTM) when the state is hidden.
- **Exploration bonuses** — count-based and curiosity methods for sparse rewards.
- **Offline RL** — learning from a fixed dataset without interacting (CQL, IQL).
- **Multi-agent RL** — self-play, cooperation, emergent behaviour.
- **RL from human feedback** — how language models are fine-tuned with a learned reward model.

## 04 · Reading list

- Sutton & Barto, *Reinforcement Learning: An Introduction* (2nd ed.) — free online; this course
  follows its structure through chapter 13.
- OpenAI *Spinning Up in Deep RL* — concise deep RL theory and clean reference implementations.
- David Silver's RL lecture series (UCL) — the classic video course.
- The CleanRL project — single-file, readable implementations of modern algorithms.

In [ ]:
#@title 🧠 **Knowledge check 1** · +25 XP — run, then pick an answer
rlcolab.quiz("14-beyond#0", {"q": "You want to control a robot arm with continuous joint torques. Which algorithm family from the table is the natural starting point?", "options": ["Tabular Q-learning", "DQN", "SAC / TD3 (continuous-control actor-critic)", "UCB bandits"], "answer": 2, "explain": "DQN needs a max over a discrete action set. Continuous actions call for actor-critic methods that output actions directly, such as SAC or TD3 (PPO works too)."})

---
### 🧪 Exercise 1 · Open sandbox — bring your own agent

A free-form file with no checks. It detects which optional libraries you have and gives you a starting point:

- With **gymnasium** installed it runs a random agent on CartPole through the real Gymnasium API.
- Without it, it falls back to the bundled `rlenvs.CartPole`.

Ideas: wrap Core Runner as a `gymnasium.Env`, train Stable-Baselines3 PPO on it, or port your DQN to PyTorch.

> 🧭 **Free-form** — no automated checks. Edit the next cell and run it (`Shift+Enter`); experiment freely.

In [ ]:
# ✏️ EXERCISE 14-beyond/01-open-sandbox
import importlib.util
import numpy as np
import rlhud

HAS = {m: importlib.util.find_spec(m) is not None for m in ("gymnasium", "torch", "stable_baselines3")}
rlhud.table([[k, "installed" if v else "—"] for k, v in HAS.items()], headers=["library", "status"], title="Optional libraries")

if HAS["gymnasium"]:
    import gymnasium as gym
    env = gym.make("CartPole-v1")
    source = "gymnasium CartPole-v1"
else:
    from rlenvs import CartPole
    env = CartPole()
    source = "bundled rlenvs.CartPole (pip install gymnasium for the real thing)"

rng = np.random.default_rng(0)
returns = []
for ep in range(50):
    obs, info = env.reset(seed=ep)
    G, done = 0.0, False
    while not done:
        obs, r, term, trunc, info = env.step(int(rng.integers(2)))
        G += r
        done = term or trunc
    returns.append(G)

rlhud.plot(returns, label="random agent", title=f"Random baseline · {source}")
rlhud.stat("Mean return", float(np.mean(returns)))
print("Your playground. Replace the random agent with something smarter!")

<details><summary>💡 <b>Hint 1</b> of 3 — try it before opening the next one</summary>

Gymnasium custom envs subclass gymnasium.Env and define observation_space, action_space, reset() and step().


<details><summary>💡 <b>Hint 2</b> of 3 — try it before opening the next one</summary>

For Core Runner level 4: observation_space = gymnasium.spaces.Box(0, 1, shape=(133,)), action_space = gymnasium.spaces.Discrete(5). Note two of the globals can be negative — use low=-1.


<details><summary>💡 <b>Hint 3</b> of 3 — nearly gives it away</summary>

from stable_baselines3 import PPO; model = PPO('MlpPolicy', env).learn(100_000)

</details>

</details>

</details>

In [ ]:
#@title 🔒 Reference solution — double-click to reveal · running it replaces your version
rlcolab.solution_gate("14-beyond/01-open-sandbox")  # obeys the Solutions setting in 00 · START HERE
# ✏️ EXERCISE 14-beyond/01-open-sandbox
# Reference solution
import importlib.util
import numpy as np
import rlhud

HAS = {m: importlib.util.find_spec(m) is not None for m in ("gymnasium", "torch", "stable_baselines3")}
rlhud.table([[k, "installed" if v else "—"] for k, v in HAS.items()], headers=["library", "status"], title="Optional libraries")

if HAS["gymnasium"]:
    import gymnasium as gym
    env = gym.make("CartPole-v1")
    source = "gymnasium CartPole-v1"
else:
    from rlenvs import CartPole
    env = CartPole()
    source = "bundled rlenvs.CartPole (pip install gymnasium for the real thing)"

rng = np.random.default_rng(0)
returns = []
for ep in range(50):
    obs, info = env.reset(seed=ep)
    G, done = 0.0, False
    while not done:
        obs, r, term, trunc, info = env.step(int(rng.integers(2)))
        G += r
        done = term or trunc
    returns.append(G)

rlhud.plot(returns, label="random agent", title=f"Random baseline · {source}")
rlhud.stat("Mean return", float(np.mean(returns)))
print("Your playground. Replace the random agent with something smarter!")

## 05 · Signing off

You started with a random drone bumping into walls. You end with agents that plan, learn from
experience and generalise to worlds they have never seen. Keep the reward signal strong.

---
## 🏁 Debrief
Run the cell below for your mission status, then open the next notebook.

In [ ]:
#@title 🏁 **Mission status** — run to refresh
rlcolab.lesson_footer("14-beyond")